# Rat K-Fold fine-tuning

Reproduce the old `3DRNET_Kfold_rats.ipynb` split protocol: five folds, training on four rats and validating/evaluating **every longitudinal scan** of the fifth. All scans of an animal stay together. D2 currently has five rats and 20 scans, including a 45-day scan; timepoint counts differ across animals.

The omitted rat also selects the checkpoint. Measurements therefore have the **validation** role, rather than an independent outer-test role. Use notebook 03 for a separate fixed train/validation/test experiment. Each fold starts from the same mouse checkpoint and a fresh optimizer.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/experiments/config.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.experiments.training import configure_runtime

TRAINING_SEED: int = 42
runtime = configure_runtime(seed=TRAINING_SEED, memory_growth=True, mixed_precision=False)
runtime
import nibabel as nib

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
reference_axes = nib.aff2axcodes(ref_img.affine)
print("Reference axes:", reference_axes, "(left-right is the R/L axis)")


## Complete rat configuration

The preset retains the old 112³ patches, 0.1 mm spacing, batch size 4, 100 epochs, learning rate 0.001 and training multiplier 5. Original model widths stay unchanged. Set `MOUSE_CHECKPOINT` to compatible complete weights or an inference model, verifying the source label meanings. Training remains disabled for the data preview.

Current validation is deterministic and covers all scans once without repetition or dropped batches. Images use linear interpolation, masks nearest-neighbour. Historical random validation augmentation and multiple inference strides are not reproduced. Full-volume evaluation retains the historical ventricle separation and brain-mask cleanup. Set `FOLD_FILE` for matched folds or `RESUME_RUN` to recover an interrupted matching run.

Full-volume outputs use `original_label_ids=True`. `postprocessing_settings` controls brain-mask cleanup and ventricle separation. Thresholds are voxel counts on the 0.1 mm grid; categorical ROI masks are not cleaned as binary masks. Merged D1/rat ventricles are separated before mapping to original IDs 3 (contra), 13 (ipsi), and 21 (third ventricle). MouseFull keeps its directly predicted sides. GT uses the selected original annotations, never the prediction postprocessor.

The historical `default_hemisphere="right"` selects the plane side containing array origin, rather than deriving anatomical right from the affine. It is used when no sufficiently large lesion is predicted. Splitting requires at least three predicted third-ventricle voxels; missing midline predictions raise an explicit scan-specific error.


In [ ]:
from src.experiments.config import (
    DataConfig, DatasetSpec, EvaluationPatchConfig, ExperimentConfig, InferenceConfig, InitializationConfig,
    LabelConfig, LossConfig, ModelConfig, PostprocessingConfig, PreprocessingConfig,
)
from src.evaluation.k_fold import EvaluationConfig, TrainingConfig

DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
MOUSE_CHECKPOINT: Path | None = None
RUN_TRAINING: bool = False
RESUME_RUN: Path | None = None
FOLD_FILE: Path | None = None

training = TrainingConfig(
    model_name="d2_3drnet_leave_one_rat_out", k=5, epochs=100, batch_size=4,
    seed=TRAINING_SEED, data_multiplier=5, output_dir=str(REPO_ROOT / "results"),
    monitor="val_loss", monitor_mode="min", default_callbacks=False,
    fit_kwargs={"verbose": 1},
)
data_settings = DataConfig(root=DATA_ROOT, datasets=(DatasetSpec(name="D2", annotation_scope="full_volume"),))
preprocessing = PreprocessingConfig(
    reference=REFERENCE, patch_size=(112, 112, 112), spacing_mm=(0.1, 0.1, 0.1),
    correct_units=True, augment=True, random_crop=True, crop_mode="center",
    # Native-grid MRI degradation; bounds retain their earlier magnitudes.
    intensity_augmentation_grid="native",
    blur_sigma_range=(0.0, 0.6), blur_sigma_units="voxels",
    noise_std_range=(0.0, 0.05), noise_std_units="intensity",
    flip_probability=0.5,
)
# Sample once; validation/test reuse evaluation_patches.json across epochs and runs.
evaluation_patch_settings = EvaluationPatchConfig(
    patches_per_scan=5, sampling="center_biased", seed=42,
    left_right_pairs=False,  # D1/rats: optional paired LR views; MouseFull: keep False.
)

model_settings = ModelConfig(
    architecture="3drnet", filters=(16, 32, 64, 128, 256),
    residual=True, attention=True, max_pooling=True, classifier=False,
)
loss_settings = LossConfig(
    regions="dice_ce", gamma=0.8, brain_alpha=0.6, batch_wise=True,
    # Total loss = weighted sum of the active output-head losses.
    regions_weight=1.0, brain_weight=1.0, classifier_weight=1.0,
)
# Full volumes: postprocess first, then restore original annotation IDs.
postprocessing_settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)
inference_settings = InferenceConfig(
    enabled=True, stride=56, save_predictions=True,
    original_label_ids=True, postprocessing=postprocessing_settings,
)
initialization = InitializationConfig()
if MOUSE_CHECKPOINT is not None:
    initialization = InitializationConfig(mode="weights", checkpoint=MOUSE_CHECKPOINT)
else:
    print("Data preview only: set MOUSE_CHECKPOINT before rat fine-tuning.")

config = ExperimentConfig(
    data=data_settings, preprocessing=preprocessing, training=training,
    evaluation_patches=evaluation_patch_settings,
    model=model_settings, labels=LabelConfig(unknown="background"), loss=loss_settings,
    inference=inference_settings, initialization=initialization, learning_rate=0.001,
    split_seed=42, cv_validation_fraction=0.2,
)
config.validate()

## Configure callbacks directly

K-Fold uses only LR reduction, with no early stopping. Add TensorBoard/CSVLogger here if you want extra logging. Edit the callback constructors or the list in the next cell; the training call explicitly receives `callbacks=callbacks`. No callback is added by a hidden experiment factory. `TrainingConfig.default_callbacks=False` prevents duplicate automatic logging/LR callbacks. Checkpoint selection uses the visible `TrainingConfig.monitor` and `monitor_mode` settings.

Relative log paths resolve inside each run/fold, so folds and study variants do not share files. The runner copies unused callback instances for every fit and saves their settings in `callbacks.json`; resume requires the same saved settings. CSV logs append when resuming.

The runner always saves best weights and epoch recovery state, even when this list is empty. It restores the selected best checkpoint before evaluation. For that reason early stopping uses `restore_best_weights=False`: restoring weights inside its epoch-end hook would mix earlier weights with the latest optimizer in the recovery checkpoint. The final best-weight behavior matches the original workflow.


In [ ]:
import tensorflow as tf

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", mode="min", factor=0.5, patience=10,
    min_lr=1e-7, verbose=1,
)
callbacks: list[tf.keras.callbacks.Callback] = [reduce_lr]


## Leave-one-rat-out coverage

Every rat supplies exactly one validation fold; every discovered timepoint is included. Saved folds must preserve this protocol.

In [ ]:
from IPython.display import display
from src.experiments.data import fold_summary, make_leave_one_animal_out_splits, prepare_data
from src.evaluation.k_fold import load_fold_splits

prepared = prepare_data(config)
cv_ids = prepared.subjects.subject_ids
if config.training.k != len(cv_ids):
    raise ValueError("Leave-one-rat-out needs one fold per registered rat")

evaluation = EvaluationConfig(
    evaluate_validation=True, save_predictions=config.inference.save_predictions,
)
if RESUME_RUN is not None:
    saved_file = RESUME_RUN / "splits.json"
    folds = load_fold_splits(saved_file, cv_ids, config.training.k)
    if FOLD_FILE is not None:
        requested_folds = load_fold_splits(FOLD_FILE, cv_ids, config.training.k)
        if requested_folds != folds:
            raise ValueError("FOLD_FILE differs from the resumed run's folds")
elif FOLD_FILE is not None:
    folds = load_fold_splits(FOLD_FILE, cv_ids, config.training.k)
else:
    folds = make_leave_one_animal_out_splits(cv_ids)

if any(fold.test_ids or len(fold.val_ids) != 1 for fold in folds):
    raise ValueError("Each rat fold must have exactly one validation rat and no outer holdout")

summary = fold_summary(folds, prepared.subjects)
manifest = prepared.subjects.manifest(cv_ids)
display(summary)
display(manifest)

## Check one validation batch

This checks the shared preprocessing and target shapes without building a model. Validation keeps its final incomplete batch and all selected timepoints.

In [ ]:
validation_ids = folds[0].val_ids
validation_generator = prepared.make_generator(validation_ids, training=False, batch_size=1)
images, targets = validation_generator[0]

print("Input:", images.shape)
print("ROI targets:", targets[0].shape)
print("Brain targets:", targets[1].shape)
print("Validation batches:", len(validation_generator))

del images, targets, validation_generator

## Train or resume

Each fold restores its best validation-selected weights before full-volume evaluation. Metrics use physical spacing; saved masks return to each input scan's original geometry and contain original annotation IDs. Configuration, annotation policy, animal manifests, splits and initialization audits are saved alongside recovery state.

In [ ]:
from src.experiments.training import run_cross_validation

result = None
if RUN_TRAINING:
    if config.initialization.mode == "scratch":
        raise ValueError("Set MOUSE_CHECKPOINT before fine-tuning rat folds")

    result = run_cross_validation(
        prepared, subject_ids=cv_ids, folds=folds,
        evaluation=evaluation, resume_run=RESUME_RUN, callbacks=callbacks,
    )
    print("Run saved to:", result.run_dir)
else:
    print("Checks complete. Set RUN_TRAINING = True to train or resume.")

## Longitudinal validation results

Results retain rat and scan identities. Multiple timepoints of one rat are not independent biological samples. The selected rat was also used for checkpoint selection.

In [ ]:
import pandas as pd
import plotly.io as pio
from src.experiments.training import history_figure

pio.renderers.default = "vscode"

if result is not None:
    figure = history_figure(result)
    figure.show()
    display(pd.DataFrame(result.timings))

    if result.evaluations.get("regions"):
        metrics = pd.concat(result.evaluations["regions"], ignore_index=True)
        evaluated = metrics.loc[metrics["DatasetRole"] == "validation"]
        dice = evaluated.loc[evaluated["Metric"] == "Dice"]
        columns = ["Fold", "Subject", "Scan", "Region", "Value", "Status"]
        display(dice[columns])
        print("Full metric tables:", result.run_dir / "regions")

## Best-checkpoint patch and full-volume measurements

Native patch losses and metrics use every saved deterministic validation/test patch. These are cohort aggregates, not independent animal scores. Full-volume brain and region scores use postprocessed predictions, original annotation IDs and valid references. The workbook contains separate sheets for the two evaluation stages.


In [ ]:
import pandas as pd

if result is not None:
    display(result.patch_metrics)
    volume_tables = [table for tables in result.evaluations.values() for table in tables]
    full_volume_metrics = pd.DataFrame()
    if volume_tables:
        full_volume_metrics = pd.concat(volume_tables, ignore_index=True)
    evaluation_folder = result.run_dir / "summary"
    selected_columns = ["DatasetRole", "Fold", "Head", "Region", "Metric", "Value", "Status"]
    if not full_volume_metrics.empty:
        display(full_volume_metrics[selected_columns].head(30))
    from src.evaluation.classification import classification_report, healthy_lesion_report
    classifier_report_result = classification_report(result.classifier_predictions)
    if not classifier_report_result.summary.empty:
        display(classifier_report_result.summary)
        display(classifier_report_result.confusion)
    sham_animals, sham_summary = healthy_lesion_report(full_volume_metrics)
    if not sham_summary.empty:
        display(sham_summary)
    print("Complete native patch and postprocessed volume tables:", evaluation_folder)
    print("Workbook:", evaluation_folder / "evaluation_metrics.xlsx")
else:
    print("Use the final best-checkpoint cell to evaluate a stopped or saved run.")


## Explore longitudinal validation predictions against ground truth

Run the cell below after training/evaluation, or point `VIEW_RUN` at an existing matching run folder to inspect it without fitting. Saved full-volume `regions.nii.gz` files are loaded directly; no predictions are recomputed. This requires inference and prediction saving to have been enabled.

The table lists every available scan/timepoint and fold. The viewer initially loads up to three rows; set `VIEW_SCAN_INDICES=(0, 4)` to explore other rows. Use the Plotly dropdown to select a scan and reference-oriented plane, then drag the slice slider. Zoom/pan works in each MRI/GT/prediction panel.

Both overlays use the same class colors. The title lists unannotated GT regions: their absence is not evidence of a prediction error. Set `VIEW_REGION` to a foreground **original annotation ID** (for example 13 for the ipsilateral ventricle) to isolate one region. `VIEW_OPACITY` controls the overlay. Previews sample up to 64 slices per plane and 192 pixels per side; set `VIEW_MAX_SLICES=None` for every slice and raise `VIEW_MAX_PIXELS` for more detail. Saved masks are unchanged. The controls use native Plotly rendering.

These rat scans also selected their fold checkpoint; they are validation predictions, not an independent test set. All timepoints retain their rat identity.


In [ ]:
from src.evaluation.inference import saved_prediction_manifest, prediction_comparison_figure

VIEW_RUN: Path | None = None  # Existing run folder, or use the result above.
VIEW_SCAN_INDICES: tuple[int, ...] | None = None  # None: first three manifest rows.
VIEW_REGION: int | None = None  # Original annotation ID (e.g. 13 = ipsi ventricle); None shows all.
VIEW_OPACITY: float = 0.55
VIEW_MAX_SLICES: int | None = 64
VIEW_MAX_PIXELS: int = 192
VIEW_ROLE: str = 'validation'

viewer_role = VIEW_ROLE
viewer_run = VIEW_RUN
current_result = globals().get("result")
if viewer_run is None and current_result is not None:
    viewer_run = current_result.run_dir
if viewer_run is None and RESUME_RUN is not None:
    viewer_run = RESUME_RUN

if viewer_run is None:
    print("Train/evaluate first, or set VIEW_RUN to an existing run folder.")
else:
    prediction_views = saved_prediction_manifest(prepared, viewer_run, role=viewer_role)
    display(prediction_views)

    if prediction_views.empty:
        print("No saved predictions for this role. Check the run, VIEW_ROLE and prediction-saving settings.")
    else:
        prediction_figure = prediction_comparison_figure(
            data=prepared, predictions=prediction_views,
            scan_indices=VIEW_SCAN_INDICES, region=VIEW_REGION,
            opacity=VIEW_OPACITY, max_slices=VIEW_MAX_SLICES,
            max_pixels=VIEW_MAX_PIXELS,
        )
        prediction_figure.show(renderer="vscode")


## Evaluate the best checkpoint after stopping training

Interrupt training, then run this cell independently of `result`. Set `CHECKPOINT_RUN` to the existing **run folder**, select `CHECKPOINT_FOLD`, and enable `RUN_CHECKPOINT_EVALUATION`. After a kernel restart, only the repository setup cell and this cell are needed: the saved experiment supplies architecture, preprocessing, label policy and animal selection.

The default cohort matches this notebook. Fixed-split runs use test animals, CV uses outer holdouts, and historical rat CV uses validation animals. Every selected animal includes all timepoints. A best checkpoint must already exist; an unfinished epoch is not evaluated. The committed selected-weight copy is used when recovery state is available.

This measures native patches and performs full-volume inference, saves metrics/predictions to a fresh folder outside the training run, and opens the MRI/GT/prediction viewer. Training weights, optimizer, history and completion flags are left intact. Inference can take time; leave the switch False to skip it. Use notebook 09 to reopen an evaluation without recomputing predictions.

The common RARE reference is explicitly used for checkpoint evaluation, including older checkpoints. A model trained with a different orientation receives a changed input convention; those scores should be interpreted as evaluation with an orientation override. Training recovery files remain unchanged.


The independent cell exposes `checkpoint_postprocessing` and always restores original annotation IDs, including when the saved run predates these settings. It writes `prediction_labels.json`, so the viewer uses the exact exported IDs. `CHECKPOINT_REGION` selects an original annotation ID. Changing evaluation settings does not change training targets or weights.


In [ ]:
from src.experiments.config import PostprocessingConfig
from src.evaluation.inference import (
    best_checkpoint_settings, evaluate_saved_model,
    saved_prediction_manifest, prediction_comparison_figure,
)

RUN_CHECKPOINT_EVALUATION: bool = False
CHECKPOINT_RUN: Path | None = None  # results/<model>/<run>; required after interruption/restart.
CHECKPOINT_FOLD: int = 0
CHECKPOINT_OUTPUT: Path | None = None  # None creates a fresh evaluation folder.
CHECKPOINT_SCAN_INDICES: tuple[int, ...] | None = None
CHECKPOINT_REGION: int | None = None
CHECKPOINT_MAX_SLICES: int | None = 64
CHECKPOINT_ROLE: str = 'validation'

checkpoint_run = CHECKPOINT_RUN
current_result = globals().get("result")
if checkpoint_run is None and current_result is not None:
    checkpoint_run = current_result.run_dir
if checkpoint_run is None:
    checkpoint_run = globals().get("RESUME_RUN")

checkpoint_postprocessing = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=20000,
    remove_small_objects=True, object_min_area=30000,
    split_ventricles=True, use_centroids=True,
    default_hemisphere="right", lesion_voxels_threshold=10,
)

checkpoint_evaluation = None
if RUN_CHECKPOINT_EVALUATION:
    if checkpoint_run is None:
        raise ValueError("Set CHECKPOINT_RUN to the stopped run folder first")
    checkpoint_settings = best_checkpoint_settings(
        run_dir=checkpoint_run, fold=CHECKPOINT_FOLD, role=CHECKPOINT_ROLE,
        output_dir=CHECKPOINT_OUTPUT, reference=REFERENCE,
        original_label_ids=True, postprocessing=checkpoint_postprocessing,
        evaluate_patches=True,
    )
    print("Selected weights:", checkpoint_settings.model_file)
    checkpoint_evaluation = evaluate_saved_model(checkpoint_settings)
    print("Evaluation saved to:", checkpoint_evaluation.output_dir)
    display(checkpoint_evaluation.patch_metrics)
    print("Evaluation workbook:", checkpoint_evaluation.output_dir / "evaluation_metrics.xlsx")
    from src.evaluation.classification import classification_report, healthy_lesion_report
    checkpoint_classifier = classification_report(checkpoint_evaluation.classifier_predictions)
    if not checkpoint_classifier.summary.empty:
        display(checkpoint_classifier.summary)
        display(checkpoint_classifier.confusion)
    checkpoint_sham_animals, checkpoint_sham_summary = healthy_lesion_report(checkpoint_evaluation.metrics)
    if not checkpoint_sham_summary.empty:
        display(checkpoint_sham_summary)
    checkpoint_metrics = checkpoint_evaluation.metrics
    checkpoint_dice = checkpoint_metrics.loc[checkpoint_metrics["Metric"] == "Dice"]
    display(checkpoint_dice)
    checkpoint_views = saved_prediction_manifest(
        data=checkpoint_evaluation.data, run_dir=checkpoint_evaluation.output_dir,
        role=CHECKPOINT_ROLE,
    )
    display(checkpoint_views)
    checkpoint_figure = prediction_comparison_figure(
        data=checkpoint_evaluation.data, predictions=checkpoint_views,
        scan_indices=CHECKPOINT_SCAN_INDICES, region=CHECKPOINT_REGION,
        max_slices=CHECKPOINT_MAX_SLICES,
    )
    checkpoint_figure.show(renderer="vscode")
else:
    print("Set CHECKPOINT_RUN and RUN_CHECKPOINT_EVALUATION=True to evaluate saved best weights.")
